# LeetCode 3024 — Type of Triangle bằng Python

Link đề: https://leetcode.com/problems/type-of-triangle/ · Độ khó: **Easy** · Chủ đề: Array, Math, Sorting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def triangleType(self, nums: List[int]) -> str:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc (đề gốc có ví dụ `[3, 3, 3]` và `[3, 4, 5]`; các ví dụ `isosceles` và `none` ở đây do mình tự dựng và đã kiểm chứng bằng code); nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho ba số là độ dài ba cạnh, trả về loại tam giác: **`"equilateral"`** (đều), **`"isosceles"`** (cân), **`"scalene"`** (thường) hoặc **`"none"`** (không tạo thành tam giác).
2. Nắm **bất đẳng thức tam giác**: ba cạnh tạo thành tam giác khi và chỉ khi **tổng hai cạnh nhỏ hơn hẳn cạnh còn lại, với cả ba cặp**; hiểu vì sao chỉ cần kiểm tra **cạnh lớn nhất**.
3. Biết **sắp xếp ba số**, dùng `set` để đếm số **giá trị khác nhau**, và thứ tự các nhánh `if` (kiểm tra **hợp lệ trước**, phân loại sau).
4. Đi từ cách **sắp xếp rồi so sánh** → **ba bất đẳng thức, không sắp xếp** → **`max`/`sum` + `len(set)`** → **tra bảng theo số giá trị khác nhau**.
5. Tránh các lỗi kinh điển: phân loại trước khi kiểm tra hợp lệ (ví dụ `[1, 1, 2]` bị gọi là cân), dùng `>=` làm tam giác suy biến thành hợp lệ, chỉ kiểm tra một bất đẳng thức, và xếp `isosceles` trước `equilateral`.

## 2. Đề bài & ví dụ

Bạn được cho một mảng số nguyên `nums` có **đúng 3 phần tử**, là độ dài ba cạnh. Hãy trả về một **chuỗi** cho biết loại tam giác tạo bởi ba cạnh đó (hoặc `"none"` nếu không tạo thành tam giác):

- `"equilateral"` (**tam giác đều**): cả **ba cạnh bằng nhau**;
- `"isosceles"` (**tam giác cân**): **đúng hai cạnh bằng nhau**;
- `"scalene"` (**tam giác thường**): **ba cạnh đôi một khác nhau**;
- `"none"`: ba cạnh **không** tạo thành một tam giác.

Ba cạnh tạo thành tam giác **khi và chỉ khi tổng hai cạnh bất kỳ lớn hơn hẳn cạnh thứ ba** (tổng bằng cạnh thứ ba là tam giác "suy biến", không tính).

**Ví dụ 1**

- Input: `nums = [3, 3, 3]`
- Output: `"equilateral"`
- Giải thích: cả ba cạnh bằng nhau (`3 + 3 > 3`, hợp lệ).

**Ví dụ 2**

- Input: `nums = [3, 4, 5]`
- Output: `"scalene"`
- Giải thích: ba cạnh khác nhau và `3 + 4 > 5`, nên là tam giác thường.

**Ví dụ 3**

- Input: `nums = [5, 5, 8]`
- Output: `"isosceles"`
- Giải thích: hai cạnh bằng nhau (`5, 5`) và `5 + 5 = 10 > 8`, hợp lệ nên là tam giác cân.

**Ví dụ 4**

- Input: `nums = [8, 4, 4]`
- Output: `"none"`
- Giải thích: `4 + 4 = 8` không lớn hơn `8` (tam giác suy biến thành đoạn thẳng), nên không phải tam giác.

**Ràng buộc:** `nums.length == 3`; `1 <= nums[i] <= 100`.

In [1]:
from typing import List
from itertools import product
import random

examples = [
    ([3, 3, 3], "equilateral"),
    ([3, 4, 5], "scalene"),
    ([5, 5, 8], "isosceles"),
    ([8, 4, 4], "none"),
    ([1, 1, 2], "none"),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", repr(expected))

nums = [3, 3, 3] -> kỳ vọng 'equilateral'
nums = [3, 4, 5] -> kỳ vọng 'scalene'
nums = [5, 5, 8] -> kỳ vọng 'isosceles'
nums = [8, 4, 4] -> kỳ vọng 'none'
nums = [1, 1, 2] -> kỳ vọng 'none'


## 3. Kiến thức nền

### 3.1. Bất đẳng thức tam giác

Ba đoạn thẳng `a, b, c` ghép thành một tam giác (có diện tích dương) khi và chỉ khi **cả ba** điều sau đúng: `a + b > c`, `a + c > b`, `b + c > a`. Nếu tổng hai cạnh **bằng** cạnh thứ ba, ba đoạn nằm trên một đường thẳng (tam giác **suy biến**, không tính là tam giác).

In [2]:
def valid_three(a, b, c):
    return a + b > c and a + c > b and b + c > a

print(valid_three(3, 4, 5))      # True
print(valid_three(4, 4, 8))      # False: 4 + 4 = 8 (suy biến)
print(valid_three(1, 2, 10))     # False

True
False
False


### 3.2. Chỉ cần kiểm tra cạnh lớn nhất

Gọi `c` là cạnh lớn nhất (`a <= b <= c`, tất cả `>= 1`). Hai bất đẳng thức còn lại **luôn đúng**: `a + c > b` (vì `c >= b` và `a >= 1` nên `a + c >= b + 1`) và `b + c > a` (vì `b >= a` và `c >= 1` nên `b + c >= a + 1`). Vì vậy chỉ còn phải kiểm tra **`a + b > c`**: tổng hai cạnh nhỏ phải lớn hơn cạnh lớn nhất.

In [3]:
a, b, c = sorted([4, 8, 4])          # [4, 4, 8]
print(a, b, c, "| a + b > c ?", a + b > c)          # 4 4 8 | False -> none
a, b, c = sorted([5, 3, 4])          # [3, 4, 5]
print(a, b, c, "| a + b > c ?", a + b > c)          # True

4 4 8 | a + b > c ? False
3 4 5 | a + b > c ? True


### 3.3. Sắp xếp ba số và giải nén (unpack)

`sorted(nums)` trả về danh sách mới tăng dần; `a, b, c = sorted(nums)` **giải nén** ba phần tử vào ba biến (đòi hỏi đúng `3` phần tử). Sau khi sắp xếp, các cạnh bằng nhau **đứng cạnh nhau**, nên chỉ cần so sánh các cặp liền kề.

In [4]:
nums = [5, 8, 5]
a, b, c = sorted(nums)
print(a, b, c)                       # 5 5 8
print(a == b, b == c, a == c)        # True False False: cạnh bằng nhau đứng kề nhau

5 5 8
True False False


### 3.4. `set` để đếm số giá trị khác nhau

`len(set(nums))` là số **giá trị khác nhau** trong `nums`: `1` giá trị thì ba cạnh bằng nhau (đều), `2` giá trị thì đúng hai cạnh bằng nhau (cân), `3` giá trị thì đôi một khác nhau (thường).

In [5]:
for nums in ([3, 3, 3], [5, 5, 8], [3, 4, 5]):
    print(nums, "->", len(set(nums)), "giá trị khác nhau")

[3, 3, 3] -> 1 giá trị khác nhau
[5, 5, 8] -> 2 giá trị khác nhau
[3, 4, 5] -> 3 giá trị khác nhau


### 3.5. Thứ tự các nhánh `if / elif`

Python chạy `if / elif` **từ trên xuống**, nhánh nào đúng trước thì dừng. Tam giác **đều cũng là cân** theo nghĩa "có ít nhất hai cạnh bằng nhau", nên điều kiện cân phải đặt **sau** điều kiện đều; còn việc "có phải tam giác không" phải được kiểm tra **trước tất cả** các phép phân loại.

In [6]:
def classify_wrong_order(a, b, c):
    if a == b or b == c or a == c:
        return "isosceles"           # [3, 3, 3] cũng vào đây trước!
    if a == b == c:
        return "equilateral"         # không bao giờ tới
    return "scalene"

print(classify_wrong_order(3, 3, 3))  # isosceles <- sai, phải là equilateral

isosceles


### 3.6. Từ điển (bảng tra) thay cho chuỗi `if`

Nếu có ánh xạ rõ ràng `số giá trị khác nhau -> tên loại`, ta có thể tra bảng: `{1: "equilateral", 2: "isosceles", 3: "scalene"}[len(set(nums))]`. Chuỗi `if` thành một dòng tra cứu.

In [7]:
names = {1: "equilateral", 2: "isosceles", 3: "scalene"}
for nums in ([3, 3, 3], [5, 5, 8], [3, 4, 5]):
    print(nums, "->", names[len(set(nums))])

[3, 3, 3] -> equilateral
[5, 5, 8] -> isosceles
[3, 4, 5] -> scalene


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hai bước: **(1) hợp lệ không?** **(2) loại nào?** (chỉ xét khi đã hợp lệ). Sắp xếp ba cạnh `a <= b <= c`:

| `nums` | sắp xếp `a, b, c` | `a + b > c`? | số cạnh bằng nhau | kết quả |
|---|---|---|---|---|
| `[3, 3, 3]` | `3, 3, 3` | `6 > 3` có | cả ba | `equilateral` |
| `[3, 4, 5]` | `3, 4, 5` | `7 > 5` có | không có | `scalene` |
| `[5, 5, 8]` | `5, 5, 8` | `10 > 8` có | hai | `isosceles` |
| `[8, 4, 4]` | `4, 4, 8` | `8 > 8` **không** | (không xét) | `none` |
| `[1, 1, 2]` | `1, 1, 2` | `2 > 2` **không** | (không xét) | `none` |

Chú ý hai dòng `none`: dù có hai cạnh bằng nhau (trông giống tam giác cân) nhưng **không hợp lệ**, nên phải trả `none`, không phải `isosceles`. Đó là lý do bước **hợp lệ phải đi trước**.

In [8]:
for nums in ([3, 3, 3], [3, 4, 5], [5, 5, 8], [8, 4, 4], [1, 1, 2]):
    a, b, c = sorted(nums)
    ok = a + b > c
    kinds = len(set(nums))
    print(f"{nums}: sắp xếp {(a, b, c)} | a+b>c: {ok!s:5} | số giá trị khác nhau: {kinds}")

[3, 3, 3]: sắp xếp (3, 3, 3) | a+b>c: True  | số giá trị khác nhau: 1
[3, 4, 5]: sắp xếp (3, 4, 5) | a+b>c: True  | số giá trị khác nhau: 3
[5, 5, 8]: sắp xếp (5, 5, 8) | a+b>c: True  | số giá trị khác nhau: 2
[8, 4, 4]: sắp xếp (4, 4, 8) | a+b>c: False | số giá trị khác nhau: 2
[1, 1, 2]: sắp xếp (1, 1, 2) | a+b>c: False | số giá trị khác nhau: 2


## 5. Cách 1 — Brute Force / baseline: sắp xếp, kiểm tra hợp lệ, rồi phân loại

**Ý tưởng.** `a, b, c = sorted(nums)`. Nếu `a + b <= c` thì `"none"`. Ngược lại nếu `a == c` (nghĩa là cả ba bằng nhau) thì `"equilateral"`; nếu `a == b` hoặc `b == c` thì `"isosceles"`; còn lại `"scalene"`.

In [9]:
def triangle_type_sort(nums: List[int]) -> str:
    a, b, c = sorted(nums)                             # a <= b <= c
    if a + b <= c:                                     # KHÔNG hợp lệ (kể cả suy biến)
        return "none"
    if a == c:                                         # a <= b <= c và a == c -> cả ba bằng nhau
        return "equilateral"
    if a == b or b == c:                               # đúng hai cạnh bằng nhau
        return "isosceles"
    return "scalene"


for nums, expected in examples:
    print(nums, "->", triangle_type_sort(nums), "(kỳ vọng", repr(expected), ")")

[3, 3, 3] -> equilateral (kỳ vọng 'equilateral' )
[3, 4, 5] -> scalene (kỳ vọng 'scalene' )
[5, 5, 8] -> isosceles (kỳ vọng 'isosceles' )
[8, 4, 4] -> none (kỳ vọng 'none' )
[1, 1, 2] -> none (kỳ vọng 'none' )


In [10]:
# Xem từng bước quyết định
for nums in ([3, 4, 5], [5, 5, 8], [8, 4, 4]):
    a, b, c = sorted(nums)
    print(f"{nums} -> sắp xếp ({a}, {b}, {c})")
    print(f"   a + b = {a + b}, c = {c}, a + b <= c ? {a + b <= c}")
    if a + b > c:
        print(f"   a == c ? {a == c} | a == b ? {a == b} | b == c ? {b == c}")

[3, 4, 5] -> sắp xếp (3, 4, 5)
   a + b = 7, c = 5, a + b <= c ? False
   a == c ? False | a == b ? False | b == c ? False
[5, 5, 8] -> sắp xếp (5, 5, 8)
   a + b = 10, c = 8, a + b <= c ? False
   a == c ? False | a == b ? True | b == c ? False
[8, 4, 4] -> sắp xếp (4, 4, 8)
   a + b = 8, c = 8, a + b <= c ? True


**Độ phức tạp (Cách 1).** Sắp xếp `3` phần tử là `O(1)` (hằng số): **Time `O(1)`**, **Space `O(1)`**.

**Nút thắt.** Đã tối ưu. Chỉ còn câu hỏi có thể tránh `sorted` hay không, và có cách phân loại gọn hơn chuỗi `if`.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét làm bài gọn hơn:

1. **Hợp lệ ⇔ tổng hai cạnh nhỏ hơn > cạnh lớn nhất ⇔ `sum(nums) > 2 * max(nums)`**. Không cần sắp xếp: tổng ba cạnh trừ cạnh lớn nhất là tổng hai cạnh còn lại, nên `sum - max > max`.
2. **Loại tam giác chỉ phụ thuộc số giá trị khác nhau** `len(set(nums))`: `1` đều, `2` cân, `3` thường (và chỉ xét khi đã hợp lệ).

Nhờ đó không cần sắp xếp và không cần so sánh từng cặp.

## 7. Các cách tiếp theo

### Cách 2 — Ba bất đẳng thức tường minh, không sắp xếp

**Giải thích.** Viết thẳng ba bất đẳng thức `a + b > c`, `a + c > b`, `b + c > a` (mục 3.1) cho `a, b, c = nums`. Nếu có cái nào sai thì `"none"`. Rồi phân loại bằng so sánh bằng nhau từng cặp.

In [11]:
def triangle_type_three(nums: List[int]) -> str:
    a, b, c = nums
    if not (a + b > c and a + c > b and b + c > a):    # đủ cả ba bất đẳng thức
        return "none"
    if a == b == c:
        return "equilateral"
    if a == b or b == c or a == c:
        return "isosceles"
    return "scalene"


for nums, expected in examples:
    print(nums, "->", triangle_type_three(nums), "(kỳ vọng", repr(expected), ")")

[3, 3, 3] -> equilateral (kỳ vọng 'equilateral' )
[3, 4, 5] -> scalene (kỳ vọng 'scalene' )
[5, 5, 8] -> isosceles (kỳ vọng 'isosceles' )
[8, 4, 4] -> none (kỳ vọng 'none' )
[1, 1, 2] -> none (kỳ vọng 'none' )


**Độ phức tạp Cách 2:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 3 — `max` / `sum` kiểm tra hợp lệ, `len(set)` phân loại

**Giải thích.** Hợp lệ khi `sum(nums) > 2 * max(nums)`. Khi đó phân loại theo `k = len(set(nums))`: `k == 1` đều, `k == 2` cân, ngược lại thường.

In [12]:
def triangle_type_set(nums: List[int]) -> str:
    if sum(nums) <= 2 * max(nums):                     # tổng hai cạnh nhỏ <= cạnh lớn nhất
        return "none"
    k = len(set(nums))                                 # số giá trị khác nhau
    if k == 1:
        return "equilateral"
    if k == 2:
        return "isosceles"
    return "scalene"


for nums, expected in examples:
    print(nums, "->", triangle_type_set(nums), "(kỳ vọng", repr(expected), ")")

[3, 3, 3] -> equilateral (kỳ vọng 'equilateral' )
[3, 4, 5] -> scalene (kỳ vọng 'scalene' )
[5, 5, 8] -> isosceles (kỳ vọng 'isosceles' )
[8, 4, 4] -> none (kỳ vọng 'none' )
[1, 1, 2] -> none (kỳ vọng 'none' )


**Độ phức tạp Cách 3:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 4 — Tra bảng theo số giá trị khác nhau

**Giải thích.** Thay chuỗi `if` phân loại bằng một **bảng tra**: danh sách `["equilateral", "isosceles", "scalene"]` đánh chỉ số theo `len(set(nums)) - 1`. Gọn và "dữ liệu hóa" quy tắc.

In [13]:
KINDS = ["equilateral", "isosceles", "scalene"]       # chỉ số = số giá trị khác nhau - 1

def triangle_type_table(nums: List[int]) -> str:
    if sum(nums) <= 2 * max(nums):
        return "none"
    return KINDS[len(set(nums)) - 1]


for nums, expected in examples:
    print(nums, "->", triangle_type_table(nums), "(kỳ vọng", repr(expected), ")")

[3, 3, 3] -> equilateral (kỳ vọng 'equilateral' )
[3, 4, 5] -> scalene (kỳ vọng 'scalene' )
[5, 5, 8] -> isosceles (kỳ vọng 'isosceles' )
[8, 4, 4] -> none (kỳ vọng 'none' )
[1, 1, 2] -> none (kỳ vọng 'none' )


**Độ phức tạp Cách 4:** **Time `O(1)`**, **Space `O(1)`**.

In [14]:
# Code nộp LeetCode cho Cách 3 (ngắn, không cần sắp xếp)
from typing import List

class Solution:
    def triangleType(self, nums: List[int]) -> str:
        if sum(nums) <= 2 * max(nums):
            return "none"
        k = len(set(nums))
        if k == 1:
            return "equilateral"
        if k == 2:
            return "isosceles"
        return "scalene"

## 8. TRACE từng bước

Ta theo dõi (a) Cách 1 và (b) Cách 3 trên nhiều bộ ba, để thấy **hợp lệ được kiểm tra trước**.

In [15]:
def trace_sort(nums):
    a, b, c = sorted(nums)
    print(f"nums={nums} -> sắp xếp ({a}, {b}, {c})")
    if a + b <= c:
        print(f"   {a} + {b} = {a + b} <= {c} -> none")
        return "none"
    print(f"   {a} + {b} = {a + b} > {c} -> hợp lệ")
    if a == c:
        print(f"   a == c ({a} == {c}) -> cả ba bằng nhau: equilateral")
        return "equilateral"
    if a == b or b == c:
        print(f"   a == b ({a == b}) hoặc b == c ({b == c}) -> isosceles")
        return "isosceles"
    print("   không có hai cạnh nào bằng nhau -> scalene")
    return "scalene"


assert trace_sort([3, 4, 5]) == "scalene"
assert trace_sort([5, 5, 8]) == "isosceles"
assert trace_sort([3, 3, 3]) == "equilateral"
assert trace_sort([1, 1, 2]) == "none"

nums=[3, 4, 5] -> sắp xếp (3, 4, 5)
   3 + 4 = 7 > 5 -> hợp lệ
   không có hai cạnh nào bằng nhau -> scalene
nums=[5, 5, 8] -> sắp xếp (5, 5, 8)
   5 + 5 = 10 > 8 -> hợp lệ
   a == b (True) hoặc b == c (False) -> isosceles
nums=[3, 3, 3] -> sắp xếp (3, 3, 3)
   3 + 3 = 6 > 3 -> hợp lệ
   a == c (3 == 3) -> cả ba bằng nhau: equilateral
nums=[1, 1, 2] -> sắp xếp (1, 1, 2)
   1 + 1 = 2 <= 2 -> none


In [16]:
def trace_set(nums):
    s, m = sum(nums), max(nums)
    print(f"nums={nums}: sum={s}, 2*max={2 * m}")
    if s <= 2 * m:
        print("   sum <= 2*max -> none")
        return "none"
    k = len(set(nums))
    print(f"   hợp lệ; len(set(nums)) = {k} -> {KINDS[k - 1]}")
    return KINDS[k - 1]


assert trace_set([8, 4, 4]) == "none"
assert trace_set([5, 5, 8]) == "isosceles"
assert trace_set([7, 7, 7]) == "equilateral"

nums=[8, 4, 4]: sum=16, 2*max=16
   sum <= 2*max -> none
nums=[5, 5, 8]: sum=18, 2*max=16
   hợp lệ; len(set(nums)) = 2 -> isosceles
nums=[7, 7, 7]: sum=21, 2*max=14
   hợp lệ; len(set(nums)) = 1 -> equilateral


Bảng tóm tắt (Cách 3):

| `nums` | `sum` | `2·max` | hợp lệ? | `len(set)` | kết quả |
|---|---|---|---|---|---|
| `[3, 3, 3]` | 9 | 6 | có | 1 | equilateral |
| `[3, 4, 5]` | 12 | 10 | có | 3 | scalene |
| `[5, 5, 8]` | 18 | 16 | có | 2 | isosceles |
| `[8, 4, 4]` | 16 | 16 | **không** (bằng) | | none |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (`sum`/`max` và `len(set)`). Đúng class `Solution`, đúng method `triangleType(self, nums: List[int]) -> str`.

In [17]:
from typing import List

class Solution:
    def triangleType(self, nums: List[int]) -> str:
        if sum(nums) <= 2 * max(nums):                 # tổng hai cạnh nhỏ <= cạnh lớn nhất: không phải tam giác
            return "none"
        k = len(set(nums))                             # số giá trị khác nhau
        if k == 1:
            return "equilateral"                       # ba cạnh bằng nhau
        if k == 2:
            return "isosceles"                         # đúng hai cạnh bằng nhau
        return "scalene"                               # ba cạnh khác nhau


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.triangleType(nums))

[3, 3, 3] -> equilateral
[3, 4, 5] -> scalene
[5, 5, 8] -> isosceles
[8, 4, 4] -> none
[1, 1, 2] -> none


## 10. Vì sao thuật toán đúng

- **Hợp lệ ⇔ `a + b > c` với `c` lớn nhất.** Nếu `c` là cạnh lớn nhất thì `a + c > b` và `b + c > a` luôn đúng (vì `c >= b >= a >= 1`), nên cả ba bất đẳng thức tam giác tương đương với riêng `a + b > c`. Do `a + b = sum - c`, điều kiện này là `sum - max > max`, tức `sum > 2 · max`. Vì vậy `sum <= 2 · max` ⇔ không phải tam giác (kể cả suy biến khi bằng).
- **Phân loại theo số giá trị khác nhau.** Với ba cạnh: `1` giá trị khác nhau là cả ba bằng nhau (đều); `2` giá trị khác nhau là đúng một cặp bằng nhau (vì ba phần tử mà chỉ có hai giá trị thì đúng hai phần tử trùng nhau); `3` là đôi một khác nhau (thường). Ba trường hợp này **loại trừ lẫn nhau và đầy đủ**.
- **Thứ tự kiểm tra.** Kiểm tra hợp lệ **trước** vì các loại đều/cân/thường chỉ có nghĩa với tam giác thật. Ví dụ `[1, 1, 2]` có hai cạnh bằng nhau nhưng không hợp lệ.
- **Tam giác đều luôn hợp lệ.** `a + a > a` với `a >= 1`; nên `[a, a, a]` không bao giờ bị trả `none`.

## 11. Phân tích Time / Space Complexity

Mảng luôn có đúng `3` phần tử, nên mọi cách đều là **hằng số**.

- **Cách 1 (sắp xếp):** Time `O(1)` (sắp xếp 3 phần tử), Space `O(1)`.
- **Cách 2 (ba bất đẳng thức):** Time `O(1)`, Space `O(1)`.
- **Cách 3 (`max`/`sum`/`set`):** Time `O(1)`, Space `O(1)`.
- **Cách 4 (tra bảng):** Time `O(1)`, Space `O(1)`.
- Ý nghĩa của bài nằm ở **tư duy kiểm tra điều kiện theo đúng thứ tự** chứ không phải tối ưu bậc.

## 12. Bộ test

In [18]:
sol = Solution()
impls = {
    "sort": triangle_type_sort,
    "three": triangle_type_three,
    "set": triangle_type_set,
    "table": triangle_type_table,
    "submit": sol.triangleType,
}

for nums, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (key, nums, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [19]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([1, 1, 1], "equilateral", "ba cạnh nhỏ nhất")
check_all([100, 100, 100], "equilateral", "ba cạnh lớn nhất")
check_all([1, 1, 2], "none", "suy biến: 1 + 1 = 2")
check_all([2, 1, 1], "none", "suy biến, đổi thứ tự")
check_all([1, 2, 1], "none", "suy biến, cạnh lớn ở giữa")
check_all([1, 2, 3], "none", "suy biến: 1 + 2 = 3, ba cạnh khác nhau")
check_all([1, 2, 4], "none", "tổng hai cạnh nhỏ < cạnh lớn")
check_all([2, 3, 4], "scalene", "tam giác thường nhỏ")
check_all([2, 2, 3], "isosceles", "cân, cạnh đáy lớn hơn")
check_all([3, 2, 2], "isosceles", "cân, cạnh đáy ở đầu")
check_all([2, 3, 2], "isosceles", "cân, cạnh đáy ở giữa")
check_all([100, 100, 1], "isosceles", "cân rất nhọn: 100, 100, 1")
check_all([100, 1, 1], "none", "một cạnh quá dài")
check_all([100, 99, 1], "none", "suy biến: 99 + 1 = 100")
check_all([100, 99, 2], "scalene", "gần suy biến nhưng hợp lệ: 99 + 2 > 100")
check_all([50, 50, 100], "none", "suy biến cân: 50 + 50 = 100")
check_all([50, 50, 99], "isosceles", "cân hợp lệ gần suy biến")

PASS ba cạnh nhỏ nhất -> equilateral
PASS ba cạnh lớn nhất -> equilateral
PASS suy biến: 1 + 1 = 2 -> none
PASS suy biến, đổi thứ tự -> none
PASS suy biến, cạnh lớn ở giữa -> none
PASS suy biến: 1 + 2 = 3, ba cạnh khác nhau -> none
PASS tổng hai cạnh nhỏ < cạnh lớn -> none
PASS tam giác thường nhỏ -> scalene
PASS cân, cạnh đáy lớn hơn -> isosceles
PASS cân, cạnh đáy ở đầu -> isosceles
PASS cân, cạnh đáy ở giữa -> isosceles
PASS cân rất nhọn: 100, 100, 1 -> isosceles
PASS một cạnh quá dài -> none
PASS suy biến: 99 + 1 = 100 -> none
PASS gần suy biến nhưng hợp lệ: 99 + 2 > 100 -> scalene
PASS suy biến cân: 50 + 50 = 100 -> none
PASS cân hợp lệ gần suy biến -> isosceles


### 12.2. Tính chất và phản ví dụ

In [20]:
# Chuẩn vàng độc lập: ba bất đẳng thức chặt + đếm cặp bằng nhau
def reference(nums):
    a, b, c = nums
    if not (a + b > c and a + c > b and b + c > a):
        return "none"
    equal_pairs = (a == b) + (b == c) + (a == c)
    return {3: "equilateral", 1: "isosceles", 0: "scalene"}[equal_pairs]


# Duyệt TOÀN BỘ các bộ ba (a, b, c) với 1 <= a, b, c <= 30
count = {"equilateral": 0, "isosceles": 0, "scalene": 0, "none": 0}
for a, b, c in product(range(1, 31), repeat=3):
    ref = reference([a, b, c])
    count[ref] += 1
    for key, fn in impls.items():
        assert fn([a, b, c]) == ref, (key, (a, b, c))
print("PASS toàn bộ 27 000 bộ ba (1..30):", count)

# Tính chất 1: hoán vị ba cạnh không đổi kết quả
random.seed(11)
for _ in range(500):
    nums = [random.randint(1, 100) for _ in range(3)]
    shuffled = nums[:]
    random.shuffle(shuffled)
    assert sol.triangleType(nums) == sol.triangleType(shuffled)
print("PASS hoán vị không đổi kết quả")

# Tính chất 2: nhân cả ba cạnh với k > 0 không đổi loại (đồng dạng)
for _ in range(500):
    nums = [random.randint(1, 30) for _ in range(3)]
    k = random.randint(2, 3)
    assert sol.triangleType(nums) == sol.triangleType([x * k for x in nums])
print("PASS tam giác đồng dạng cùng loại")

# Tính chất 3: "equilateral" <=> ba số bằng nhau; "none" <=> sum <= 2*max
for _ in range(500):
    nums = [random.randint(1, 5) for _ in range(3)]
    r = sol.triangleType(nums)
    assert (r == "equilateral") == (len(set(nums)) == 1)
    assert (r == "none") == (sum(nums) <= 2 * max(nums))
print("PASS equilateral <=> bằng nhau; none <=> sum <= 2 * max")

# Tính chất 4: đầu vào không bị sửa
nums = [5, 4, 3]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: phân loại trước khi kiểm tra hợp lệ
def sai_khong_kiem_tra_hop_le(nums):
    a, b, c = nums
    if a == b == c:
        return "equilateral"
    if a == b or b == c or a == c:
        return "isosceles"
    return "scalene"

assert sai_khong_kiem_tra_hop_le([1, 1, 2]) == "isosceles" and sol.triangleType([1, 1, 2]) == "none"
print("PASS phản ví dụ: [1, 1, 2] bị gọi là isosceles nếu không kiểm tra hợp lệ")

# Phản ví dụ 2: dùng >= (chấp nhận suy biến)
def sai_ge(nums):
    if sum(nums) < 2 * max(nums):
        return "none"
    k = len(set(nums))
    return KINDS[k - 1]

assert sai_ge([8, 4, 4]) == "isosceles" and sol.triangleType([8, 4, 4]) == "none"
print("PASS phản ví dụ: chấp nhận tổng bằng nhau cho", sai_ge([8, 4, 4]), "thay vì none")

PASS toàn bộ 27 000 bộ ba (1..30): {'equilateral': 30, 'isosceles': 1935, 'scalene': 11550, 'none': 13485}
PASS hoán vị không đổi kết quả
PASS tam giác đồng dạng cùng loại
PASS equilateral <=> bằng nhau; none <=> sum <= 2 * max
PASS đầu vào không bị sửa
PASS phản ví dụ: [1, 1, 2] bị gọi là isosceles nếu không kiểm tra hợp lệ
PASS phản ví dụ: chấp nhận tổng bằng nhau cho isosceles thay vì none


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [21]:
random.seed(3024)
for trial in range(5000):
    nums = [random.randint(1, 100) for _ in range(3)]
    if random.random() < 0.3:
        nums[random.randrange(3)] = nums[random.randrange(3)]          # tăng khả năng có cạnh bằng nhau
    ref = reference(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 5000 bộ ngẫu nhiên trên miền 1..100 - mọi cách khớp chuẩn vàng")

PASS 5000 bộ ngẫu nhiên trên miền 1..100 - mọi cách khớp chuẩn vàng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Phân loại trước khi kiểm tra hợp lệ

In [22]:
def sai_thu_tu_1(nums):
    a, b, c = nums
    if a == b == c:
        return "equilateral"
    if a == b or b == c or a == c:
        return "isosceles"                           # [1, 1, 2] vào đây
    if a + b <= c or a + c <= b or b + c <= a:       # kiểm tra hợp lệ đến quá muộn
        return "none"
    return "scalene"


print("sai  :", sai_thu_tu_1([1, 1, 2]), "| đúng:", Solution().triangleType([1, 1, 2]))

sai  : isosceles | đúng: none


**Vì sao sai:** `[1, 1, 2]` có hai cạnh bằng nhau nhưng `1 + 1 = 2` nên **không** là tam giác. Vì nhánh `isosceles` đứng trước phép kiểm tra hợp lệ nên bị trả sai. Luôn kiểm tra **hợp lệ trước** rồi mới phân loại.

### Lỗi 2: Dùng `>=` (chấp nhận tam giác suy biến)

In [23]:
def sai_ge_2(nums):
    a, b, c = sorted(nums)
    if a + b < c:                                    # nên là a + b <= c
        return "none"
    if a == c:
        return "equilateral"
    if a == b or b == c:
        return "isosceles"
    return "scalene"


print("sai  :", sai_ge_2([8, 4, 4]), "| đúng:", Solution().triangleType([8, 4, 4]))

sai  : isosceles | đúng: none


**Vì sao sai:** đề đòi tổng hai cạnh **lớn hơn hẳn** cạnh thứ ba. Khi bằng nhau (`4 + 4 = 8`) ba đoạn nằm trên một đường thẳng, không phải tam giác. Điều kiện không hợp lệ là `a + b <= c` (có dấu bằng).

### Lỗi 3: Chỉ kiểm tra một bất đẳng thức khi chưa sắp xếp

In [24]:
def sai_mot_bat_dang_thuc(nums):
    a, b, c = nums                                   # CHƯA sắp xếp
    if a + b <= c:
        return "none"
    k = len(set(nums))
    return KINDS[k - 1]


print("sai  :", sai_mot_bat_dang_thuc([1, 10, 1]), "| đúng:", Solution().triangleType([1, 10, 1]))

sai  : isosceles | đúng: none


**Vì sao sai:** chỉ cần kiểm tra `a + b > c` **khi `c` là cạnh lớn nhất**. Với `[1, 10, 1]` (chưa sắp xếp) ta kiểm tra `1 + 10 > 1` (đúng) và bỏ qua rằng `10` mới là cạnh lớn nhất (`1 + 1 < 10`). Hoặc sắp xếp trước, hoặc kiểm tra đủ cả ba bất đẳng thức, hoặc dùng `sum > 2 * max`.

### Lỗi 4: Xếp `isosceles` trước `equilateral`

In [25]:
def sai_thu_tu_2(nums):
    if sum(nums) <= 2 * max(nums):
        return "none"
    a, b, c = nums
    if a == b or b == c or a == c:
        return "isosceles"                           # [3, 3, 3] cũng thỏa "có hai cạnh bằng nhau"
    if a == b == c:
        return "equilateral"
    return "scalene"


print("sai  :", sai_thu_tu_2([3, 3, 3]), "| đúng:", Solution().triangleType([3, 3, 3]))

sai  : isosceles | đúng: equilateral


**Vì sao sai:** tam giác đều cũng có (ít nhất) hai cạnh bằng nhau, nên `a == b or b == c or a == c` đúng với `[3, 3, 3]` và nhánh `isosceles` chặn mất nhánh `equilateral`. Hãy kiểm tra trường hợp **đặc biệt hơn** (cả ba bằng nhau) **trước**, hoặc dùng `len(set(nums))` để chia ba trường hợp loại trừ nhau.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "ba cạnh có tạo thành tam giác không" → **bất đẳng thức tam giác**: sắp xếp rồi `a + b > c`, hoặc `sum > 2 * max`.
- Công thức nhớ: **hợp lệ ⇔ `sum(nums) > 2 * max(nums)`** (nghiêm ngặt: bằng nhau là suy biến).
- Phân loại theo số **giá trị khác nhau**: `len(set(nums))` bằng `1`, `2`, `3` ứng với đều, cân, thường.
- **Thứ tự kiểm tra:** hợp lệ trước, rồi trường hợp **đặc biệt nhất** (đều) trước trường hợp tổng quát hơn (cân).
- Với bài có nhiều nhánh điều kiện, hãy **liệt kê toàn bộ** các bộ ba nhỏ (như `1..30`) và so với một chuẩn vàng độc lập.
- Tránh so sánh số thực trong bài này; dữ liệu là số nguyên nên các phép so sánh chính xác.

## 15. Biến thể của bài

1. **Diện tích** tam giác bằng công thức Heron.
2. **Tam giác vuông**: `a² + b² == c²` (với `c` lớn nhất).
3. **Tam giác nhọn / vuông / tù** theo bình phương cạnh lớn nhất.
4. Từ một mảng `n` cạnh, **đếm số bộ ba** tạo thành tam giác (bài "Valid Triangle Number").
5. **Chu vi lớn nhất** của tam giác từ một mảng cạnh (bài "Largest Perimeter Triangle").

Thử biến thể 1, 2, 3 và 5:

In [26]:
from math import sqrt

def heron_area(nums):
    if Solution().triangleType(nums) == "none":
        return 0.0
    a, b, c = nums
    s = (a + b + c) / 2
    return sqrt(s * (s - a) * (s - b) * (s - c))


def angle_type(nums):
    a, b, c = sorted(nums)
    if a + b <= c:
        return "none"
    if a * a + b * b == c * c:
        return "right"
    return "acute" if a * a + b * b > c * c else "obtuse"


def largest_perimeter(arr):
    arr = sorted(arr, reverse=True)
    for i in range(len(arr) - 2):
        if arr[i] < arr[i + 1] + arr[i + 2]:               # arr[i] là cạnh lớn nhất của bộ ba
            return arr[i] + arr[i + 1] + arr[i + 2]
    return 0


print("1. diện tích 3-4-5:", heron_area([3, 4, 5]))                      # 6.0
print("3. loại góc:", angle_type([3, 4, 5]), angle_type([4, 5, 6]), angle_type([2, 3, 4]), angle_type([1, 1, 3]))
print("5. chu vi lớn nhất của [2, 1, 2]:", largest_perimeter([2, 1, 2]), "| [1, 2, 1, 10]:", largest_perimeter([1, 2, 1, 10]))
assert heron_area([3, 4, 5]) == 6.0 and heron_area([1, 1, 3]) == 0.0
assert [angle_type(t) for t in ([3, 4, 5], [4, 5, 6], [2, 3, 4], [1, 1, 3])] == ["right", "acute", "obtuse", "none"]
assert largest_perimeter([2, 1, 2]) == 5 and largest_perimeter([1, 2, 1, 10]) == 0 and largest_perimeter([3, 2, 3, 4]) == 10

1. diện tích 3-4-5: 6.0
3. loại góc: right acute obtuse none
5. chu vi lớn nhất của [2, 1, 2]: 5 | [1, 2, 1, 10]: 0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp xếp rồi `a + b <= c` | cạnh lớn nhất ở cuối | `O(1)` | `O(1)` | dễ hiểu, giống định nghĩa |
| 2. Ba bất đẳng thức tường minh | không sắp xếp | `O(1)` | `O(1)` | khi muốn bám sát định lý |
| 3. `sum > 2·max` + `len(set)` | không cần sắp xếp, ba trường hợp loại trừ | `O(1)` | `O(1)` | **cách nộp bài**, ngắn gọn |
| 4. Tra bảng theo `len(set) - 1` | dữ liệu hóa phân loại | `O(1)` | `O(1)` | khi có nhiều loại cần ánh xạ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sorted([5, 2, 4])`, `len(set([4, 4, 7]))`, `Solution().triangleType([7, 7, 7])`, `Solution().triangleType([2, 2, 5])`, `Solution().triangleType([4, 6, 9])`.

**Bài 2 (tính tay).** Với từng bộ ba: `[6, 6, 10]`, `[5, 12, 13]`, `[3, 3, 6]`, `[9, 4, 4]`, `[7, 8, 2]` hãy sắp xếp, kiểm tra `a + b > c` và cho biết loại tam giác.

**Bài 3 (điền TODO).** Hoàn thành `triangle_type_todo(nums)` theo Cách 3.

In [27]:
def triangle_type_todo(nums):
    # TODO 1: nếu sum(nums) <= 2 * max(nums) thì trả về "none"
    # TODO 2: k = len(set(nums)); k == 1 -> "equilateral", k == 2 -> "isosceles", còn lại "scalene"
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `triangle_type_sort2(nums)` theo **Cách 1** (sắp xếp, `a + b <= c`, so sánh `a == c`, `a == b or b == c`) mà không nhìn code mẫu. Kiểm tra với năm ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_triangles(arr)`: số bộ ba **chỉ số** `i < j < k` mà `arr[i], arr[j], arr[k]` tạo thành tam giác. (b) Viết `is_right_triangle(nums)`: `True` nếu là tam giác **vuông**. (c) Viết `smallest_missing_side(a, b)`: số nguyên dương **nhỏ nhất** `c` sao cho `[a, b, c]` **tạo được tam giác**.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[2, 4, 5]`, `2`, `"equilateral"`, `"none"` (`2 + 2 = 4 < 5`), `"scalene"` (`4 + 6 = 10 > 9`, ba cạnh khác nhau). Bài 3: hai TODO như trong comment.
- **Mức 2 (vừa):** Bài 2: `[6, 6, 10]`: `12 > 10` hợp lệ, hai cạnh bằng nhau: isosceles. `[5, 12, 13]`: `17 > 13`: scalene. `[3, 3, 6]`: `6 > 6`? không: none. `[9, 4, 4]`: sắp xếp `4, 4, 9`: `8 > 9`? không: none. `[7, 8, 2]`: sắp xếp `2, 7, 8`: `9 > 8` có: scalene. Bài 4: xem code Cách 1.
- **Mức 3 (sâu):** Bài 5(a): sắp xếp rồi hai con trỏ (hoặc vét cạn `O(n³)` cho đúng). 5(b): `a, b, c = sorted(nums)`; vuông khi `a*a + b*b == c*c`. 5(c): hợp lệ cần `|a - b| < c < a + b`, nên `c` nhỏ nhất là `|a - b| + 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from itertools import combinations

# Bài 1
assert sorted([5, 2, 4]) == [2, 4, 5] and len(set([4, 4, 7])) == 2
assert Solution().triangleType([7, 7, 7]) == "equilateral"
assert Solution().triangleType([2, 2, 5]) == "none"
assert Solution().triangleType([4, 6, 9]) == "scalene"

# Bài 2
assert [Solution().triangleType(t) for t in ([6, 6, 10], [5, 12, 13], [3, 3, 6], [9, 4, 4], [7, 8, 2])] == \
       ["isosceles", "scalene", "none", "none", "scalene"]

# Bài 3
def triangle_type_todo_ans(nums):
    if sum(nums) <= 2 * max(nums):
        return "none"
    k = len(set(nums))
    if k == 1:
        return "equilateral"
    if k == 2:
        return "isosceles"
    return "scalene"

for nums, expected in examples:
    assert triangle_type_todo_ans(nums) == expected

# Bài 4
def triangle_type_sort2(nums):
    a, b, c = sorted(nums)
    if a + b <= c:
        return "none"
    if a == c:
        return "equilateral"
    if a == b or b == c:
        return "isosceles"
    return "scalene"

for nums, expected in examples:
    assert triangle_type_sort2(nums) == expected

# Bài 5(a)
def count_triangles(arr):
    return sum(1 for t in combinations(arr, 3) if sum(t) > 2 * max(t))

assert count_triangles([2, 2, 3, 4]) == 3
assert count_triangles([4, 2, 3, 4]) == 4
assert count_triangles([1, 1, 3]) == 0

# Bài 5(b)
def is_right_triangle(nums):
    a, b, c = sorted(nums)
    return a + b > c and a * a + b * b == c * c

assert is_right_triangle([5, 3, 4]) is True
assert is_right_triangle([3, 4, 6]) is False
assert is_right_triangle([1, 1, 2]) is False

# Bài 5(c)
def smallest_missing_side(a, b):
    return abs(a - b) + 1

for a in range(1, 15):
    for b in range(1, 15):
        c = smallest_missing_side(a, b)
        assert Solution().triangleType([a, b, c]) != "none"
        if c > 1:
            assert Solution().triangleType([a, b, c - 1]) == "none"
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hợp lệ trước, rồi phân loại theo số giá trị khác nhau)

```python
from typing import List

class Solution:
    def triangleType(self, nums: List[int]) -> str:
        if sum(nums) <= 2 * max(nums):
            return "none"
        k = len(set(nums))
        if k == 1:
            return "equilateral"
        if k == 2:
            return "isosceles"
        return "scalene"
```

**Checklist**

- [ ] Kiểm tra **hợp lệ trước** (`sum > 2 * max`, hoặc `a + b > c` sau khi sắp xếp)?
- [ ] Điều kiện không hợp lệ có dấu `=` (`sum <= 2 * max`), vì suy biến không tính là tam giác?
- [ ] Kiểm tra `equilateral` **trước** `isosceles` (hoặc dùng `len(set)` loại trừ lẫn nhau)?
- [ ] Chuỗi kết quả đúng chính tả: `"equilateral"`, `"isosceles"`, `"scalene"`, `"none"` (chữ thường)?
- [ ] Đã thử: `[1, 1, 2]`, `[8, 4, 4]`, `[100, 100, 1]`, `[3, 3, 3]`, bộ ba chưa sắp xếp?
- [ ] Skeleton đúng `class Solution` / `def triangleType(self, nums: List[int]) -> str`?